## Test the /search endpoints for AUXIP products
### Test Cases:
#### 1. "search" - using the /auxip/search endpoint
#### 2. "filter" - using the /auxip/collections/collection_id/items endpoint
#### 3. "filter" - pairs of properties - using the /auxip/collections/collection_id/items endpoint
#### 4. "sort" with /auxip/search endpoint
#### 5. "sort" with /auxip/collections/collection_id/items endpoint

In [ ]:
from pystac_client.exceptions import APIError
# Init environment before running a demo notebook.
from resources.utils import *  
init_demo()
# Reload the global vars again
from resources.utils import * 

# Counters for total requests and failures
total_requests = 0
total_failures = 0
failed_requests = []

adgs_search_properties = {
    "Name": "S1A_OPER_MPL_ORBSCT_20240514T150704_99999999T999999_0025",
    "datetime": "2024-02-14T02:08:13.372Z",
    "created": "2024-02-14T02:08:13.372Z",
    "start_datetime": "2024-02-14T02:08:13.372Z",
    "end_datetime": "2024-02-14T02:08:23.372Z",
    "platform": "sentinel-1a",
    "constellation": "sentinel-1",
    "published": "2024-02-14T02:08:13.372Z",
    "product:type": "OPER_AUX_RESORB_OPOD",
    "processing:datetime": "2024-02-14T00:00:00.000Z",
    "processing:facility": "FOS"
}

adgs_sortable_properties = ["id", "auxip:id", "file:size", "type", "eviction_datetime", "created", "start_datetime", "end_datetime"]

adgs_collection = "adgs"


In [ ]:
# 1. "search" - using the /auxip/search endpoint

for prop, value in adgs_search_properties.items():
    filter=f"{prop}='{value}'"
    print(f"filter: {filter}")
    total_requests += 1
    try:
        items_collection_adgs = auxip_client.search(method="GET", stac_filter=filter)                
        print(f"✅ RsClient request succeded for {prop} ({value}): {len(items_collection_adgs)} items")
    except RuntimeError as e:
        print(f"❌ RsClient failed for {prop}: {e}")
        total_failures += 1
        failed_requests.append(str(e))
    print("-" * 80)

In [ ]:
# 2. "filter" - using the /auxip/search endpoint but now the collection name will be present in the filter

for prop, value in adgs_search_properties.items():
    filter=f"{prop}='{value}'"
    print(f"filter: {filter}")
    total_requests += 1    
    try:
        items_collection_adgs = auxip_client.search(method="GET", 
                                                    collections = [adgs_collection],
                                                    stac_filter=filter)        
        print(f"✅ RsClient request succeded for {prop} ({value}): {len(items_collection_adgs)} items")        
    except RuntimeError as e:
        print(f"❌ RsClient failed for {prop}: {e}")
        total_failures += 1
        failed_requests.append(str(e))
    print("-" * 80)

In [ ]:
# 3. "filter" by pairs of properties - using the /auxip/collections/<id>/items endpoint

# Generate all possible pairs of search properties
keys = list(adgs_search_properties.keys())
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        prop1, prop2 = keys[i], keys[j]
        value1, value2 = adgs_search_properties[prop1], adgs_search_properties[prop2]
        filter=f"{prop1}='{value1}' AND {prop2}='{value2}'"
        print(f"filter = {filter}")        
        total_requests += 1
        try:
            items_collection_adgs = auxip_client.search(method="GET", 
                                                    collections = [adgs_collection],
                                                    stac_filter=filter)        
            print(f"✅ RsClient request succeded for {prop} ({value}): {len(items_collection_adgs)} items")            
        except RuntimeError as e:
            if not "Too complex" in str(e):
                # Too complex is allowed since it's comming from mockup
                print(f"❌ Request failed for {prop1} and {prop2}: {e}")
                total_failures += 1
                failed_requests.append(str(e))
        print("-" * 80)

In [ ]:
# 4. "sort" with "search" /auxip/search endpoint": requests for each search property with sorting and response limit

# for prop, value in adgs_search_properties.items():
#     formatted_value = f'"{value}"'
#     for sort_prop in adgs_sortable_properties:
#         url = f"{BASE_URL_SEARCH}?filter={prop}={formatted_value}&sortby=-{sort_prop}&limit=5"
#         print(f"Requesting: {url}")
#         total_requests += 1
        
#         try:
#             response = http_session.get(url)
#             response.raise_for_status()
#             data = response.json()
            
#             # Count how many items are in "features"
#             num_items = len(data.get("features", []))
#             print(f"✅ Request succeded for  {prop} ({value}) sorted by {sort_prop}: {num_items} items")
#         except requests.exceptions.RequestException as e:
#             print(f"❌ Request failed for {prop} sorted by {sort_prop}: {e}")
#             total_failures += 1
#             failed_requests.append(str(e))
#         print("-" * 80)

In [ ]:
# 5. "filter" with "search" /auxip/collections/collection_id/items endpoint": requests for each search property with sorting and response limit

# for prop, value in adgs_search_properties.items():
#     formatted_value = f'"{value}"'
#     for sort_prop in adgs_sortable_properties:
#         url = f"{BASE_URL_FILTER}?filter={prop}={formatted_value}&sortby=-{sort_prop}&limit=5"
#         print(f"Requesting: {url}")
#         total_requests += 1
        
#         try:
#             response = http_session.get(url)
#             response.raise_for_status()
#             data = response.json()
            
#             # Count how many items are in "features"
#             num_items = len(data.get("features", []))
#             print(f"✅ Request succeded for  {prop} ({value}) sorted by {sort_prop}: {num_items} items")
#         except requests.exceptions.RequestException as e:
#             print(f"❌ Request failed for {prop} sorted by {sort_prop}: {e}")
#             total_failures += 1
#             failed_requests.append(str(e))
#         print("-" * 80)

In [ ]:
# ****************************** Summary report ******************************
print("*" * 30, "Summary report", "*" * 30)
print(f"Total requests: {total_requests}")
print(f"Total failed requests: {total_failures}")
if failed_requests:
    print("Failed requests details:")
    for error in failed_requests:
        print(error)

assert total_failures == 0
assert not failed_requests